# Credit Card Default Prediction

This notebook loads the dataset from the Kaggle source (`uciml/default-of-credit-card-clients-dataset`) and trains/tests a baseline model to predict whether a customer will default on the next payment cycle.

## 1) Load dataset

The dataset (`uciml/default-of-credit-card-clients-dataset`) is downloaded from Kaggle with `kagglehub`. If Kaggle is unreachable, the notebook falls back to the local Excel file at `data/dataset.xls`. Either way the result is a DataFrame named `data`.

In [ ]:
# Import libraries and dependencies
# (install with: pip install -r requirements.txt)

from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
# Load the dataset from Kaggle with kagglehub (cached locally after the first download).
# If Kaggle can't be reached, fall back to the copy committed at data/dataset.xls.
dataset_slug = "uciml/default-of-credit-card-clients-dataset"
local_path = Path("data") / "dataset.xls"


def load_from_kaggle(slug):
    # Imported here so a missing kagglehub install falls back to the local file
    # instead of breaking the whole notebook.
    import kagglehub

    folder = Path(kagglehub.dataset_download(slug))
    csv_files = sorted(folder.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV file found in {folder}")
    frame = pd.read_csv(csv_files[0])
    # The Kaggle CSV names the target "default.payment.next.month"; the rest of this
    # notebook (and the UCI workbook) uses "default payment next month".
    frame.columns = frame.columns.str.strip().str.replace(".", " ", regex=False)
    return frame, f"Kaggle ({csv_files[0].name})"


def load_from_local(path):
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path.resolve()}")
    # The first row holds the original field names; the workbook header is on row 2.
    return pd.read_excel(path, sheet_name="Data", header=1, engine="xlrd"), str(path)


try:
    data, source = load_from_kaggle(dataset_slug)
except Exception as err:
    print(f"Kaggle download failed ({type(err).__name__}: {err}); using local copy instead.")
    data, source = load_from_local(local_path)

print(f"Loaded {data.shape[0]:,} rows and {data.shape[1]:,} columns from {source}")
display(data.head())

## 2) Prepare training data

In [ ]:
# Feature key (UCI "default of credit card clients"). Amounts are in NT dollars;
# monthly columns run from September 2005 (suffix 0/1) back to April 2005 (suffix 6).
months = ["September", "August", "July", "June", "May", "April"]
feature_key = {
    "ID": "ID of each client",
    "LIMIT_BAL": "Amount of given credit (individual + family/supplementary credit)",
    "SEX": "Gender (1=male, 2=female)",
    "EDUCATION": "1=graduate school, 2=university, 3=high school, 4=others, 5/6=unknown",
    "MARRIAGE": "Marital status (1=married, 2=single, 3=others)",
    "AGE": "Age in years",
}
for i, month in zip([0, 2, 3, 4, 5, 6], months):
    feature_key[f"PAY_{i}"] = f"Repayment status in {month} 2005 (-1=pay duly, 1..9=months of delay)"
for i, month in enumerate(months, start=1):
    feature_key[f"BILL_AMT{i}"] = f"Bill statement amount in {month} 2005"
for i, month in enumerate(months, start=1):
    feature_key[f"PAY_AMT{i}"] = f"Amount of previous payment in {month} 2005"
feature_key["default payment next month"] = "Target: default payment (1=yes, 0=no)"

missing = set(feature_key) - set(data.columns)
assert not missing, f"Columns missing from dataset: {missing}"

# Feature groups used by the preprocessing pipeline.
target_column = "default payment next month"
id_column = "ID"
categorical_features = ["SEX", "EDUCATION", "MARRIAGE"]
repayment_status_features = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
bill_amount_features = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_amount_features = [f"PAY_AMT{i}" for i in range(1, 7)]
numeric_features = ["LIMIT_BAL", "AGE"] + bill_amount_features + payment_amount_features

pd.DataFrame.from_dict(feature_key, orient="index", columns=["description"])

In [ ]:
from sklearn.model_selection import train_test_split

clean = data.copy()

# The data contains codes the feature key doesn't document:
# EDUCATION 0 (and 5/6 "unknown") and MARRIAGE 0 are folded into "others".
clean["EDUCATION"] = clean["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
clean["MARRIAGE"] = clean["MARRIAGE"].replace({0: 3})
# PAY_* also uses -2 (no consumption) and 0 (revolving credit) beyond the documented -1..9;
# they are kept as-is since the scale stays ordinal.

# ID is an identifier, not a predictor.
X = clean.drop(columns=[id_column, target_column])
Y = clean[target_column]

# Stratify so both splits keep the ~22% default rate.
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, stratify=Y, random_state=42
)

print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows | {X.shape[1]} features")
print(f"Default rate - train: {Y_train.mean():.3f}, test: {Y_test.mean():.3f}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Categorical codes are one-hot encoded; repayment status is ordinal (months of delay),
# so it is scaled alongside the amounts rather than one-hot encoded.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", StandardScaler(), numeric_features + repayment_status_features),
    ]
)

## 3) Train and evaluate baseline model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline

# Baseline: logistic regression; class_weight offsets the ~78/22 class imbalance.
baseline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000)),
    ]
)
baseline.fit(X_train, Y_train)

Y_pred = baseline.predict(X_test)
Y_proba = baseline.predict_proba(X_test)[:, 1]
print(classification_report(Y_test, Y_pred, target_names=["no default", "default"]))
print(f"ROC AUC: {roc_auc_score(Y_test, Y_proba):.3f}")